# Testing State Formations Suppressed Data Handling Methods using Quarterly State Formation Data

## Setup

In [11]:
# Packages
import pandas as pd
import numpy as np

# Loading Data
quarterly = pd.read_csv('../data_intermediate/quarterly_state_formations.csv')


In [12]:
# Quick Cleanup
quarterly = quarterly.drop(columns = ['geo_idx', 'sa', 'series'])
print(quarterly.head())

   Unnamed: 0 STATE  state_fips STATE_NAME  year    Q1    Q2    Q3    Q4  \
0           0    AL           1    Alabama  2005  1851  1664  1474  1503   
1           1    AL           1    Alabama  2006  1749  1559  1437  1302   
2           2    AL           1    Alabama  2007  1655  1414  1297  1155   
3           3    AL           1    Alabama  2008  1279  1171  1051   898   
4           4    AL           1    Alabama  2009  1113   952   943   858   

   state_formations  
0              6492  
1              6047  
2              5521  
3              4399  
4              3866  


In [ ]:
# Creating dataset with missing values
quarters = ['Q1', 'Q2', 'Q3', 'Q4']
missing = quarterly.copy()

for quarter in quarters:
    missing[f'partial_{quarter}'] = missing[quarter]/3
    # partial_quarter variables represent one third of the quarter total formations

print(missing.head())

   Unnamed: 0 STATE  state_fips STATE_NAME  year    Q1    Q2    Q3    Q4  \
0           0    AL           1    Alabama  2005  1851  1664  1474  1503   
1           1    AL           1    Alabama  2006  1749  1559  1437  1302   
2           2    AL           1    Alabama  2007  1655  1414  1297  1155   
3           3    AL           1    Alabama  2008  1279  1171  1051   898   
4           4    AL           1    Alabama  2009  1113   952   943   858   

   state_formations  partial_Q1  partial_Q2  partial_Q3  partial_Q4  
0              6492  617.000000  554.666667  491.333333  501.000000  
1              6047  583.000000  519.666667  479.000000  434.000000  
2              5521  551.666667  471.333333  432.333333  385.000000  
3              4399  426.333333  390.333333  350.333333  299.333333  
4              3866  371.000000  317.333333  314.333333  286.000000  


## Testing Annualization Method
annualized = (12/10)xincomplete

100 x (raw-annualized)/raw


### *Computing Annualizations*

In [ ]:
# Creating Annualized Formations

for missing_quarter in quarters:
    # Creating annual sums with one partially missing quarter for each quarter
    missing[f'miss_{missing_quarter}_sum'] = missing[quarters].sum(axis=1) - missing[missing_quarter] + missing[f'partial_{missing_quarter}']
    
    # Creating annualized total for each partially missing quarter sum
    missing[f'missing_{missing_quarter}_ann'] = missing[f'miss_{missing_quarter}_sum']*(12/10)

   Unnamed: 0 STATE  state_fips STATE_NAME  year    Q1    Q2    Q3    Q4  \
0           0    AL           1    Alabama  2005  1851  1664  1474  1503   
1           1    AL           1    Alabama  2006  1749  1559  1437  1302   
2           2    AL           1    Alabama  2007  1655  1414  1297  1155   
3           3    AL           1    Alabama  2008  1279  1171  1051   898   
4           4    AL           1    Alabama  2009  1113   952   943   858   

   state_formations  ...  partial_Q3  partial_Q4  miss_Q1_sum  miss_Q2_sum  \
0              6492  ...  491.333333  501.000000  5258.000000  5382.666667   
1              6047  ...  479.000000  434.000000  4881.000000  5007.666667   
2              5521  ...  432.333333  385.000000  4417.666667  4578.333333   
3              4399  ...  350.333333  299.333333  3546.333333  3618.333333   
4              3866  ...  314.333333  286.000000  3124.000000  3231.333333   

   miss_Q3_sum  miss_Q4_sum  missing_Q1_ann  missing_Q2_ann  missing_Q3_an

In [32]:
# Calculating % Difference
for quarter in quarters:
    # Calculates proportion difference between raw and annualized formations
    missing[f'ann_{quarter}_diff'] = ((missing[f'missing_{quarter}_ann'] - missing['state_formations'])/missing['state_formations']) * 100

print(missing.head())

   Unnamed: 0 STATE  state_fips STATE_NAME  year    Q1    Q2    Q3    Q4  \
0           0    AL           1    Alabama  2005  1851  1664  1474  1503   
1           1    AL           1    Alabama  2006  1749  1559  1437  1302   
2           2    AL           1    Alabama  2007  1655  1414  1297  1155   
3           3    AL           1    Alabama  2008  1279  1171  1051   898   
4           4    AL           1    Alabama  2009  1113   952   943   858   

   state_formations  ...  miss_Q3_sum  miss_Q4_sum  missing_Q1_ann  \
0              6492  ...  5509.333333  5490.000000          6309.6   
1              6047  ...  5089.000000  5179.000000          5857.2   
2              5521  ...  4656.333333  4751.000000          5301.2   
3              4399  ...  3698.333333  3800.333333          4255.6   
4              3866  ...  3237.333333  3294.000000          3748.8   

   missing_Q2_ann  missing_Q3_ann  missing_Q4_ann  ann_Q1_diff  ann_Q2_diff  \
0          6459.2          6611.2          

### *Evaluating Method*

#### **For All States**


In [45]:
for quarter in quarters:    
# Printing results over all states
    print(f'{quarter} ANNUALIZED DIFFERENCES:')
    print('mean difference:')
    print(missing[f'ann_{quarter}_diff'].mean())
# Printing describe output for each quarter
    print(missing[f'ann_{quarter}_diff'].describe())
    print('--------')
    print(
        missing[f'ann_{quarter}_diff']
        .abs()
        .quantile([.5, .9, .95, .99])
    )
    print('--------')

Q1 ANNUALIZED DIFFERENCES:
mean difference:
-2.557992354046051
count    510.000000
mean      -2.557992
std        1.063589
min       -6.556837
25%       -3.280046
50%       -2.581934
75%       -1.892132
max        0.952381
Name: ann_Q1_diff, dtype: float64
--------
0.50    2.581934
0.90    3.844074
0.95    4.220717
0.99    4.985250
Name: ann_Q1_diff, dtype: float64
--------
Q2 ANNUALIZED DIFFERENCES:
mean difference:
-1.0151868401118445
count    510.000000
mean      -1.015187
std        1.081193
min       -4.476576
25%       -1.461362
50%       -0.900220
75%       -0.346672
max        1.503759
Name: ann_Q2_diff, dtype: float64
--------
0.50    0.938747
0.90    2.616220
0.95    3.320909
0.99    3.994499
Name: ann_Q2_diff, dtype: float64
--------
Q3 ANNUALIZED DIFFERENCES:
mean difference:
1.537427956766217
count    510.000000
mean       1.537428
std        0.889392
min       -1.278962
25%        0.992431
50%        1.539759
75%        2.053902
max        5.146199
Name: ann_Q3_diff, dtyp

#### **For High Suppression States**

In [48]:
high_sup_states = ['AK', 'DC', 'ND', 'RI', 'SD', 'WV', 'WY', 'HI', 'NH']

missing_sup_states = missing[missing['STATE'].isin(high_sup_states)]

for quarter in quarters:    
# Printing results over all states
    print(f'{quarter} ANNUALIZED DIFFERENCES:')
    print('mean difference:')
    print(missing_sup_states[f'ann_{quarter}_diff'].mean())
# Printing describe output for each quarter
    print(missing_sup_states[f'ann_{quarter}_diff'].describe())
    print('--------')
    print(
        missing_sup_states[f'ann_{quarter}_diff']
        .abs()
        .quantile([.5, .9, .95, .99])
    )
    print('--------')


Q1 ANNUALIZED DIFFERENCES:
mean difference:
-2.2968846585434552
count    90.000000
mean     -2.296885
std       1.308333
min      -4.986301
25%      -3.338533
50%      -2.343443
75%      -1.362163
max       0.952381
Name: ann_Q1_diff, dtype: float64
--------
0.50    2.343443
0.90    3.882678
0.95    4.307887
0.99    4.817444
Name: ann_Q1_diff, dtype: float64
--------
Q2 ANNUALIZED DIFFERENCES:
mean difference:
-1.6564341369118307
count    90.000000
mean     -1.656434
std       1.393098
min      -4.476576
25%      -2.773827
50%      -1.680287
75%      -0.787940
max       1.503759
Name: ann_Q2_diff, dtype: float64
--------
0.50    1.680287
0.90    3.475324
0.95    3.952452
0.99    4.306709
Name: ann_Q2_diff, dtype: float64
--------
Q3 ANNUALIZED DIFFERENCES:
mean difference:
1.6099773456749615
count    90.000000
mean      1.609977
std       1.156691
min      -1.278962
25%       0.883572
50%       1.493764
75%       2.306650
max       4.869867
Name: ann_Q3_diff, dtype: float64
--------
0.

## Conclusion

For now, from this quick analysis, I am going to annualize state-year observations with 2 or less missing months and drop state-year observations with more than 2 missing months. After the preliminary analysis, I will look more into testing the validity of this annualization more rigorously.